# CSS repetition-code decoding

Select one CSS sector of a reduced GKP repetition code. The notebook compares a single LSD and quantized-PDF decode, then performs a reproducible logical-failure sweep.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

In [ ]:
d = 3
bit_flip = false
code_definition = GKP_Rep_Code(d, bit_flip, true)
M = code_definition.code
H = bit_flip ? M[1:d, 1:d] : M[(d + 1):end, (d + 1):end]
G = round.(sqrt(2) .* inv(H)) ./ sqrt(2)
logical_check = inv(H)
problem = QuantumDecodingProblem(H, G, logical_check)

sigma = 0.08
rng = MersenneTwister(20)
error_vector = sample_error(rng, sigma, size(H, 2))
received = copy(error_vector)

lsd_decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=size(H, 2),
)
lsd_estimate = run_decoder!(lsd_decoder, received)
lsd_decision = hard_decision(lsd_estimate, H)
lsd_residual = error_vector - (received - G * lsd_decision)

quantized_decoder = LDLCDecoder(
    initialize_tanner_graph_quant(H; L=128, Δ=1 / 32, widen=true);
    schedule=:parallel,
    sigma,
    max_iterations=2,
)
quantized_estimate = run_decoder!(quantized_decoder, received)
quantized_decision = hard_decision(quantized_estimate, H)
quantized_residual = error_vector - (received - G * quantized_decision)

(
    lsd_logical_error=is_logical_error(logical_check, lsd_residual),
    quantized_logical_error=is_logical_error(logical_check, quantized_residual),
)

In [ ]:
sigmas = [0.06, 0.09, 0.12]
local_search = LocalSearch(size(G, 2), G, [1])
estimates = [
    estimate_logical_error_rate!(
        MersenneTwister(200),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=size(H, 2),
        ),
        problem;
        samples=samples_per_point,
        local_search,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        failures=result.events,
        samples=result.samples,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

The serial estimator is deterministic for a fixed environment and seed. Cluster orchestration and output-file management intentionally remain outside the package API.